# Mamba-YOLO — YOLOv12s adaptation, custom dataset
Bật **Internet + GPU**. Notebook này chỉ train bản **mamba**. Mô hình giữ YOLOv12s và thêm module theo ý tưởng tác giả; không phải tái lập nguyên kiến trúc/kết quả paper. Dataset do bạn Add Input. Chạy từng cell và dừng nếu có lỗi.

In [ ]:
import os, sys, subprocess
from pathlib import Path
REPO_URL = "https://github.com/mluu59990-collab/Mamba-YOLO.git"
ROOT = Path("/kaggle/working/Mamba-YOLO")
if not ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
else:
    print("Dùng checkout hiện có; không tự ghi đè thay đổi hoặc kết quả train.")
os.chdir(ROOT)
subprocess.run(["git", "rev-parse", "HEAD"], check=True)
subprocess.run(["bash", "scripts/setup_kaggle.sh"], check=True)

## 1. Dataset của bạn
Sửa DATA_ROOT, TRAIN, VAL, NAMES theo dữ liệu đã Add Input. Nhãn YOLO detection là `class xc yc w h` chuẩn hóa 0..1, thư mục labels song song images. Với Roboflow dùng `train/images` và `valid/images`. Không tự đổi split test/val.

In [ ]:
import yaml
DATA_ROOT = Path("/kaggle/input/YOUR_DATASET")  # SỬA
TRAIN = "images/train"                        # SỬA
VAL = "images/val"                            # SỬA
NAMES = {0: "object"}                         # SỬA toàn bộ tên lớp
assert DATA_ROOT.is_dir(), "Chọn dataset trong Add Input và sửa DATA_ROOT"
assert (DATA_ROOT / TRAIN).is_dir(), DATA_ROOT / TRAIN
assert (DATA_ROOT / VAL).is_dir(), DATA_ROOT / VAL
assert list(sorted(NAMES)) == list(range(len(NAMES))), "Class ID phải liên tiếp từ 0"
DATA = Path("/kaggle/working/custom-data.yaml")
DATA.write_text(yaml.safe_dump(dict(path=str(DATA_ROOT), train=TRAIN, val=VAL, names=NAMES), allow_unicode=True))
print(DATA.read_text())

## 2. Huấn luyện mamba
Mặc định 300 epochs, 640 pixels, SGD; batch **4** là điều chỉnh cho Kaggle, không phải batch nguyên bản tác giả. Giảm batch nếu thiếu VRAM. Bản Mamba dùng lịch LR từ source Mamba-YOLO; bản AKC dùng LR cố định sau warmup theo paper AKC. Để đối chứng, dùng baseline cùng cấu hình trong repo này.

In [ ]:
EPOCHS = 300
BATCH = 4
IMGSZ = 640
SEED = 0
PROJECT = Path("/kaggle/working/runs-mamba")
subprocess.run([sys.executable, "train.py", "--variant", "mamba", "--profile", "author",
                "--data", str(DATA), "--epochs", str(EPOCHS), "--batch", str(BATCH),
                "--imgsz", str(IMGSZ), "--seed", str(SEED), "--device", "0",
                "--project", str(PROJECT), "--name", "mamba"], check=True)

## 3. Xem kết quả và giữ checkpoint
Save Version với outputs để giữ weights/best.pt, last.pt, args.yaml, environment.json, results.csv. Khi tên run trùng, framework tự tăng hậu tố. Kết quả phụ thuộc dataset; không dùng số mAP trong paper làm kết quả cho mô hình này.

In [ ]:
import pandas as pd
for result in sorted(PROJECT.glob("*/results.csv")):
    print(result)
    display(pd.read_csv(result).tail(1))
for weight in sorted(PROJECT.glob("*/weights/*.pt")):
    print(weight)

## 4. Resume ở phiên sau (tùy chọn)
Add Input output của phiên trước. Bỏ dấu # và sửa đường dẫn last.pt. Resume chỉ tiếp tục run chưa kết thúc theo lịch đã lưu. Với phiên mới chạy cell setup và dataset, rồi cell resume; không chạy cell train mới trước đó.

In [ ]:
# LAST = "/kaggle/input/YOUR_PREVIOUS_OUTPUT/last.pt"
# subprocess.run([sys.executable, "train.py", "--resume", LAST, "--data", str(DATA), "--device", "0"], check=True)

## 5. Đánh giá checkpoint tốt nhất (tùy chọn)
Sửa đường dẫn best.pt theo run của bạn rồi bỏ dấu #.

In [ ]:
# BEST = "/kaggle/working/runs-mamba/mamba/weights/best.pt"
# subprocess.run([sys.executable, "train.py", "--task", "val", "--variant", "mamba",
#                 "--weights", BEST, "--data", str(DATA), "--batch", str(BATCH), "--device", "0"], check=True)